# Chapter 3 Lab — Visualizing Light Cone Overlap

**Basal Cognition** · Dr. Ernesto Lee · [BasalCognition.com](https://basalcognition.com)

---

## What you will build

In this lab you will:

1. **Represent** a cognitive light cone as a simple 2D shape (spatial reach × temporal reach)
2. **Plot** two light cones on the same axes — one for a human decision-maker, one for an AI agent
3. **Calculate** the overlap area (the "aligned zone")
4. **Simulate** what happens as you shrink or grow one cone — how does alignment change?
5. **Run a behavioral test battery** (delayed gratification, scope-of-repair, goal persistence) and map results onto the cone

No biology background required. You need Python and matplotlib.

---

**Core idea to keep in mind while coding:**  
Alignment is not a yes/no property. It is a region. The bigger the overlap, the more the agent is actually optimizing for what you want. The smaller the overlap, the more you have reward hacking — even if neither party is doing anything "wrong."

In [ ]:
# Install dependencies
!pip install matplotlib numpy shapely --quiet

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from shapely.geometry import Polygon

# Color palette matching the book's style
TEAL   = '#0d9488'
GREEN  = '#84cc16'
SLATE  = '#64748b'
ORANGE = '#f97316'
WHITE  = '#f8fafc'

print('Imports OK. Ready to plot light cones.')

## Part 1 — Define a Light Cone

We model a light cone as an ellipse on a 2D plane:
- **X-axis**: Spatial reach (0 = none, 10 = global)
- **Y-axis**: Temporal reach (0 = immediate, 10 = decades/centuries)

An ellipse has two radii: `rx` (spatial) and `ry` (temporal).

A bacterium might be `rx=0.1, ry=0.1`. A human is `rx=7, ry=8`. An institution is `rx=9, ry=9`.

In [ ]:
def make_ellipse_polygon(cx, cy, rx, ry, n_points=200):
    """Return a Shapely Polygon approximating an ellipse."""
    theta = np.linspace(0, 2 * np.pi, n_points)
    x = cx + rx * np.cos(theta)
    y = cy + ry * np.sin(theta)
    return Polygon(zip(x, y))

def plot_two_cones(human_rx, human_ry, agent_rx, agent_ry,
                  human_label='Human Goal', agent_label='AI Agent',
                  title='Cognitive Light Cone Overlap'):
    """
    Plot two light cones and calculate their overlap.
    Both cones are centered at the origin.
    Returns the overlap fraction (0–1) relative to the human cone.
    """
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.set_facecolor(WHITE)
    fig.patch.set_facecolor(WHITE)

    # Build Shapely polygons for overlap calculation
    human_poly = make_ellipse_polygon(0, 0, human_rx, human_ry)
    agent_poly = make_ellipse_polygon(0, 0, agent_rx, agent_ry)
    overlap_poly = human_poly.intersection(agent_poly)

    # Draw human cone
    human_patch = patches.Ellipse((0, 0), width=2*human_rx, height=2*human_ry,
                                   color=TEAL, alpha=0.35, zorder=2)
    ax.add_patch(human_patch)
    human_edge = patches.Ellipse((0, 0), width=2*human_rx, height=2*human_ry,
                                  fill=False, edgecolor=TEAL, linewidth=2.5, zorder=3)
    ax.add_patch(human_edge)

    # Draw agent cone
    agent_patch = patches.Ellipse((0, 0), width=2*agent_rx, height=2*agent_ry,
                                   color=SLATE, alpha=0.3, zorder=2)
    ax.add_patch(agent_patch)
    agent_edge = patches.Ellipse((0, 0), width=2*agent_rx, height=2*agent_ry,
                                  fill=False, edgecolor=SLATE, linewidth=2.5,
                                  linestyle='--', zorder=3)
    ax.add_patch(agent_edge)

    # Draw overlap region
    if not overlap_poly.is_empty and overlap_poly.geom_type == 'Polygon':
        ox, oy = overlap_poly.exterior.xy
        ax.fill(ox, oy, color=GREEN, alpha=0.55, zorder=4, label='Aligned zone')

    # Labels
    ax.text(human_rx * 0.6, human_ry * 0.85, human_label,
            color=TEAL, fontsize=12, fontweight='bold', ha='center')
    ax.text(agent_rx * 0.6, -agent_ry * 0.75, agent_label,
            color=SLATE, fontsize=12, fontweight='bold', ha='center')

    overlap_frac = overlap_poly.area / human_poly.area if human_poly.area > 0 else 0
    ax.set_title(f'{title}\nAlignment = {overlap_frac:.1%} of human cone covered',
                 fontsize=14, pad=12)

    ax.set_xlabel('Spatial Reach →', fontsize=12)
    ax.set_ylabel('Temporal Reach →', fontsize=12)
    ax.set_xlim(-11, 11)
    ax.set_ylim(-11, 11)
    ax.axhline(0, color=SLATE, linewidth=0.5, alpha=0.4)
    ax.axvline(0, color=SLATE, linewidth=0.5, alpha=0.4)
    ax.set_aspect('equal')
    plt.tight_layout()
    plt.show()
    return overlap_frac

# Demo: a well-aligned agent
overlap = plot_two_cones(human_rx=7, human_ry=8,
                         agent_rx=6, agent_ry=7,
                         title='Well-Aligned Agent')
print(f'Overlap fraction: {overlap:.1%}')

## Part 2 — Simulate Light Cone Collapse

Watch what happens to alignment as the agent's cone shrinks.

This is the cancer / reward-hacking scenario in numbers.

In [ ]:
# Sweep agent cone size from full to collapsed
scale_factors = np.linspace(1.0, 0.05, 20)  # 100% down to 5% of human cone
human_rx, human_ry = 7.0, 8.0
overlaps = []

for s in scale_factors:
    agent_rx = human_rx * s
    agent_ry = human_ry * s
    human_poly = make_ellipse_polygon(0, 0, human_rx, human_ry)
    agent_poly  = make_ellipse_polygon(0, 0, agent_rx, agent_ry)
    overlap = human_poly.intersection(agent_poly).area / human_poly.area
    overlaps.append(overlap)

fig, ax = plt.subplots(figsize=(9, 5))
ax.set_facecolor(WHITE)
fig.patch.set_facecolor(WHITE)
ax.plot(scale_factors * 100, [o * 100 for o in overlaps],
        color=TEAL, linewidth=2.5, marker='o', markersize=4)
ax.fill_between(scale_factors * 100, [o * 100 for o in overlaps],
                alpha=0.2, color=GREEN)
ax.set_xlabel('Agent cone size (% of human cone)', fontsize=12)
ax.set_ylabel('Alignment (% of human cone covered)', fontsize=12)
ax.set_title('Alignment Collapses as the Agent Light Cone Shrinks', fontsize=13)
ax.axhline(50, color=ORANGE, linestyle='--', linewidth=1.5, label='50% alignment threshold')
ax.legend()
plt.tight_layout()
plt.show()

## Part 3 — Behavioral Test Battery

Record results from the three light-cone tests (delayed gratification, scope-of-repair, goal-persistence) and map them onto the cone diagram.

**Instructions:**
- Run each test (described below) on a system of your choice — a simple RL agent, an LLM assistant, even a human.
- Score each test 0–10.
- The scores become the spatial and temporal radii of the cone.

**Test design (fill these in for your system):**

| Test | What you measure | Score (0–10) |
|------|-----------------|-------------|
| Delayed gratification | How many steps ahead does the system hold out for a better reward? | ??? |
| Scope of repair | Does it fix only local damage, or does it restore the full correct structure? | ??? |
| Goal persistence | Does it reroute around obstacles to reach the same outcome? | ??? |

In [ ]:
# TODO: Replace these scores with your actual test results
# -----------------------------------------------------------
# System A = the system you are testing (e.g., GPT-4o, a RL agent, a student group)
# System B = the human benchmark or the ideal system

system_a_label = 'AI Agent (your system here)'
system_b_label = 'Human benchmark'

# Scores 0–10 for each test
# spatial_reach = average of scope-of-repair score
# temporal_reach = average of delayed-gratification + goal-persistence

# System A (TODO: fill in your scores)
a_delayed_grat   = 3   # How many steps ahead?
a_scope_repair   = 4   # Local only (0) to full structure (10)?
a_goal_persist   = 5   # Gives up immediately (0) to reroutes fully (10)?

# System B (human benchmark)
b_delayed_grat   = 8
b_scope_repair   = 7
b_goal_persist   = 8

# Derive cone radii from scores
a_rx = a_scope_repair          # spatial
a_ry = (a_delayed_grat + a_goal_persist) / 2  # temporal

b_rx = b_scope_repair
b_ry = (b_delayed_grat + b_goal_persist) / 2

print(f'{system_a_label}: spatial={a_rx:.1f}, temporal={a_ry:.1f}')
print(f'{system_b_label}: spatial={b_rx:.1f}, temporal={b_ry:.1f}')

overlap = plot_two_cones(human_rx=b_rx, human_ry=b_ry,
                         agent_rx=a_rx, agent_ry=a_ry,
                         human_label=system_b_label,
                         agent_label=system_a_label,
                         title='Light Cone Overlap from Behavioral Tests')
print(f'\nAlignment score: {overlap:.1%} of human cone covered')

## Deliverable

Complete the cell below. Write 150–250 words answering:

1. What system did you test?
2. What scores did it get on each behavioral test, and why?
3. What does the overlap diagram tell you about where the alignment gap is?
4. Based on the light cone framework: is this a spatial failure, a temporal failure, or both? What would "restoring the channel" look like for this system?

Submit this notebook with the deliverable cell filled in.

### Your Response

*(Replace this text with your 150–250 word response.)*

**System tested:**  

**Behavioral test scores and reasoning:**  

**What the overlap diagram shows:**  

**Spatial, temporal, or both — and how to restore the channel:**  